# HydroSense-Kenya – Level 3: Core Numerical Methods Engine
**ICS 2207 Scientific Computing | February – May 2026**
---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numerical_methods import (bisection, newton_raphson, secant,
                                forward_difference, backward_difference, central_difference,
                                trapezoidal_rule, simpsons_rule, gaussian_elimination)

plt.rcParams.update({'figure.dpi':120,'axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False,'font.size':11})

weather = pd.read_csv("../data/raw/weather_daily.csv", na_values=["NA",""])
weather['date'] = pd.to_datetime(weather['date'])
weather['rainfall_mm'].fillna(0.0, inplace=True)
weather['humidity_pct'].fillna(weather['humidity_pct'].median(), inplace=True)
T = weather['temperature_c'].values
W = weather['wind_speed_mps'].values
Solar = weather['solar_index'].values
H = weather['humidity_pct'].values
et_arr = np.maximum(0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)
weather['et_mm'] = et_arr
print("Setup complete.")

## 2. Root Finding – Irrigation Amount to Reach Target Moisture

We need to find how much irrigation `I` (mm) brings Zone_A from current moisture `S_t` to target `S_target`.

Define: `f(I) = S_t + 0.1*(R + I) - ET - S_target = 0`

In [ ]:
# Zone A parameters
S_t      = 27.9   # current soil moisture (%) – 13 Mar reading
R        = 0.0    # no rain forecast
ET       = 4.5    # typical ET
S_target = 33.0   # target moisture
FC       = 41.0   # field capacity

def f_irrigation(I):
    """How much irrigation closes the moisture deficit?"""
    return S_t + 0.1*(R + I) - ET - S_target

def df_irrigation(I):
    """Derivative of f w.r.t. I (constant)."""
    return 0.1

# Expected solution: I = (S_target - S_t + ET - 0.1*R) / 0.1
I_exact = (S_target - S_t + ET - 0.1*R) / 0.1
print(f"Exact solution: I = {I_exact:.4f} mm")
print(f"Verification: f({I_exact:.4f}) = {f_irrigation(I_exact):.2e}\n")

r_bi = bisection(f_irrigation, 0, 200, tol=1e-6)
r_nr = newton_raphson(f_irrigation, df_irrigation, x0=30.0, tol=1e-6)
r_sc = secant(f_irrigation, x0=20.0, x1=50.0, tol=1e-6)

print(f"{'Method':<18} {'Root (mm)':>12} {'Iterations':>12} {'Error':>12} {'Converged':>10}")
print("-"*66)
for name, r in [('Bisection', r_bi), ('Newton-Raphson', r_nr), ('Secant', r_sc)]:
    print(f"{name:<18} {r['root']:>12.6f} {r['iterations']:>12} {r['error']:>12.2e} {str(r['converged']):>10}")

In [ ]:
# Convergence comparison plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, r, color) in zip(
    [axes[0], axes[0], axes[0]],
    [('Bisection', r_bi, 'steelblue'), ('Newton-Raphson', r_nr, 'firebrick'), ('Secant', r_sc, '#5aab61')]
):
    if r['errors']:
        axes[0].semilogy(range(1, len(r['errors'])+1), r['errors'], 'o-', color=color, label=name, lw=2, ms=5)

axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Error (log scale)')
axes[0].set_title('Convergence of Root-Finding Methods', fontweight='bold')
axes[0].legend()

# Bar: iterations to convergence
methods = ['Bisection','Newton-Raphson','Secant']
iters   = [r_bi['iterations'], r_nr['iterations'], r_sc['iterations']]
colors  = ['steelblue','firebrick','#5aab61']
axes[1].bar(methods, iters, color=colors, alpha=0.8)
axes[1].set_ylabel('Iterations to convergence')
axes[1].set_title('Iterations Comparison', fontweight='bold')
for i, v in enumerate(iters):
    axes[1].text(i, v+0.1, str(v), ha='center', fontweight='bold')

plt.suptitle('Figure 5: Root-Finding Method Convergence', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/fig5_root_finding.png', dpi=150, bbox_inches='tight')
plt.show()
print("Newton-Raphson converges fastest (quadratic convergence) but requires derivative.")
print("Bisection is slowest (linear) but most robust — guaranteed to converge.")

## 3. Numerical Differentiation – Rate of Soil Moisture Change

In [ ]:
soil = pd.read_csv("../data/raw/soil_sensor_data.csv", na_values=["NA",""])
soil['timestamp'] = pd.to_datetime(soil['timestamp'])
zone_a = soil[soil['zone_id']=='Zone_A'].sort_values('timestamp').copy()
sm_values = zone_a['soil_moisture_pct'].fillna(zone_a['soil_moisture_pct'].median()).values

# Use finite differences on discrete soil moisture time series
def sm_func(t_idx):
    idx = int(np.round(np.clip(t_idx, 0, len(sm_values)-1)))
    return sm_values[idx]

print(f"{'Day':>4} {'SM(%)':>8} {'Forward':>10} {'Backward':>10} {'Central':>10}")
print("-"*46)
for t in range(1, len(sm_values)-1):
    fwd = (sm_values[t+1] - sm_values[t])    / 1.0
    bwd = (sm_values[t]   - sm_values[t-1])  / 1.0
    cen = (sm_values[t+1] - sm_values[t-1])  / 2.0
    print(f"{t+1:>4} {sm_values[t]:>8.2f} {fwd:>10.4f} {bwd:>10.4f} {cen:>10.4f}")

In [ ]:
# Plot moisture change rates
days = np.arange(2, len(sm_values))
fwd_rates = np.diff(sm_values)[1:]          # forward diff (day 2 onward)
cen_rates = [(sm_values[i+1]-sm_values[i-1])/2 for i in range(1, len(sm_values)-1)]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(zone_a['timestamp'].values, sm_values, 'o-', color='#e07b39', lw=2)
axes[0].set_ylabel('Soil Moisture (%)'); axes[0].set_title('Zone_A Soil Moisture', fontweight='bold')

axes[1].plot(zone_a['timestamp'].values[1:-1], fwd_rates, 'o-', color='steelblue', lw=1.5, label='Forward diff')
axes[1].plot(zone_a['timestamp'].values[1:-1], cen_rates, 's--', color='firebrick', lw=1.5, label='Central diff')
axes[1].axhline(0, color='black', lw=0.8, ls=':')
axes[1].set_ylabel('dS/dt (%/day)'); axes[1].legend()
axes[1].set_title('Rate of Moisture Change (Finite Differences)', fontweight='bold')

plt.suptitle('Figure 6: Soil Moisture Differentiation – Zone_A', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/fig6_differentiation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Interpretation: Negative rates indicate drying days; positive rates indicate recharge days.")
print("Central differences are more accurate than forward/backward (O(h²) vs O(h) error).")

## 4. Numerical Integration – Cumulative Water Deficit

In [ ]:
# Compute daily deficit for Zone_A: ET - effective rainfall contribution
rain = weather['rainfall_mm'].fillna(0).values
deficit_daily = np.maximum(0, et_arr - rain * 0.1)  # daily unmet ET

dx = 1.0   # 1 day spacing
cumulative_trap = np.cumsum([trapezoidal_rule(deficit_daily[:i+1], dx) for i in range(len(deficit_daily))])
cumulative_simp_vals = []
for i in range(len(deficit_daily)):
    y = deficit_daily[:i+1]
    if len(y) >= 3 and len(y) % 2 == 1:
        cumulative_simp_vals.append(simpsons_rule(y, dx))
    else:
        cumulative_simp_vals.append(np.nan)

total_trap = trapezoidal_rule(deficit_daily, dx)
total_simp = simpsons_rule(deficit_daily if len(deficit_daily)%2==1 else deficit_daily[:-1], dx)
print(f"Total 30-day water deficit:")
print(f"  Trapezoidal rule: {total_trap:.4f} mm-days")
print(f"  Simpson's rule:   {total_simp:.4f} mm-days")
print(f"  NumPy reference:  {np.trapz(deficit_daily, dx=dx):.4f} mm-days")
print(f"  Difference (trap vs simp): {abs(total_trap-total_simp):.6f}  (Simpson more accurate)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
dates = weather['date'].values
axes[0].bar(dates, deficit_daily, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Daily Deficit (mm)')
axes[0].set_title('Daily Water Deficit', fontweight='bold')

axes[1].plot(dates, cumulative_trap, 'o-', color='firebrick', lw=2, label='Trapezoidal')
simp_clean = [(d, v) for d, v in zip(dates, cumulative_simp_vals) if not np.isnan(v)]
if simp_clean:
    axes[1].plot([x[0] for x in simp_clean], [x[1] for x in simp_clean],
                 's--', color='#5aab61', lw=2, label="Simpson's", ms=4)
axes[1].set_xlabel('Date'); axes[1].set_ylabel('Cumulative Deficit (mm-days)')
axes[1].set_title('Cumulative Water Deficit', fontweight='bold'); axes[1].legend()

plt.suptitle("Figure 7: Numerical Integration of Water Deficit – March 2026", fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/fig7_integration.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Linear Systems – Three-Zone Water Allocation

In [ ]:
# Problem: Allocate total daily pump capacity across three zones.
# Constraints (simplified):
#   Zone_A needs 1.5x Zone_C; Zone_B needs 1.2x Zone_C
#   Total irrigation = 60 mm/day (pump capacity)
#   Zone_A area = 120 m², Zone_B = 90 m², Zone_C = 180 m²

# System: A x = b  where x = [I_A, I_B, I_C] in mm
A_mat = np.array([
    [1.0,  1.0,  1.0 ],   # Total: I_A + I_B + I_C = 60
    [1.0,  0.0, -1.5 ],   # I_A = 1.5 * I_C
    [0.0,  1.0, -1.2 ],   # I_B = 1.2 * I_C
])
b_vec = np.array([60.0, 0.0, 0.0])

x_manual = gaussian_elimination(A_mat, b_vec)
x_numpy  = np.linalg.solve(A_mat, b_vec)

print("Three-Zone Water Allocation  (Ax = b)")
print(f"  Manual Gaussian elimination: I_A={x_manual[0]:.4f}, I_B={x_manual[1]:.4f}, I_C={x_manual[2]:.4f} mm")
print(f"  NumPy reference:             I_A={x_numpy[0]:.4f},  I_B={x_numpy[1]:.4f},  I_C={x_numpy[2]:.4f} mm")
print(f"  Max absolute error vs NumPy: {np.max(np.abs(x_manual - x_numpy)):.2e}")
print(f"\n  Verification (A @ x_manual): {A_mat @ x_manual}")
print(f"  Expected b:                  {b_vec}")
print(f"\nInterpretation: Zone_A (tomato) receives {x_manual[0]:.1f} mm, Zone_B (kale) {x_manual[1]:.1f} mm,")
print(f"Zone_C (maize) {x_manual[2]:.1f} mm. Maize gets least per unit because of its larger drainage area.")

## 6. Summary
| Deliverable | Status |
|---|---|
| Bisection, Newton-Raphson, Secant (manual) | ✅ |
| Convergence comparison table & plot | ✅ |
| Forward, backward, central finite differences | ✅ |
| Trapezoidal & Simpson integration | ✅ |
| Gaussian elimination + 3-zone water allocation | ✅ |

**Key finding:** Newton-Raphson converged in the fewest iterations. Simpson's rule is more accurate than the trapezoidal rule. Gaussian elimination exactly matches NumPy's solver to machine precision.